In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_20_try_5.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['last_ones']
me:  20
trying: ['tqdm']
me:  0
trying: ['sample_submission']
me:  2
trying: ['myword']
me:  12
trying: ['train_first_last']
me:  11
trying: ['len_dict']
me:  12
trying: ['cols_to_display']
me:  14
trying: ['word_dict']
me:  12
trying: ['counter']
me:  11
trying: ['glob']
me:  0
trying: ['mylen']
me:  12
trying: ['ax']
me:  11
trying: ['IREWR_plug_2']
me:  16
trying: ['train_first']
me:  11
trying: ['train_last']
me:  11
trying: ['train']
me:  14
trying: ['stopwords']
me:  1
trying: ['plt']
me:  0
trying: ['add_gap_rows_2']
me:  22
trying: ['style']
me:  0
trying: ['CountVectorizer']
me:  0
trying: ['sample_discourse_id']
me:  2
trying: ['FuncFormatter']
me:  0
trying: ['warnings']
me:  0
trying: ['pd']
me:  0
trying: ['cols_to_merge']
me:  13
trying: ['ax2']
me:  8
trying: ['ax1']
me:  8
trying: ['np']
me:  0
trying: ['orig_output']
me:  21
trying: ['test_txt']
me:  1
trying: ['av_per_e

In [4]:
%%RecordEvent
%%time
### cell 21 ###

# lowercase the discourse text (runs on GPU via the cudf.pandas extension)
train['discourse_text'] = train['discourse_text'].str.lower()

# build the stopword list
stop_english = stopwords.words('english')
other_words_to_take_out = ['school', 'students', 'people', 'would', 'could', 'many']
stop_english.extend(other_words_to_take_out)

# split each discourse_text into words and explode so each word gets its own row
exploded = train.assign(Word=train['discourse_text'].str.split()).explode('Word')

# filter out stopwords (runs on GPU)
filtered = exploded[~exploded['Word'].isin(stop_english)]

# count frequencies of each word by discourse_type
freq = (
    filtered
    .groupby(['discourse_type', 'Word'])
    .size()
    .reset_index(name='Frequency')
)

# for each discourse_type, take the top 10 words by Frequency
top10 = (
    freq
    .sort_values(['discourse_type', 'Frequency'], ascending=[True, False])
    .groupby('discourse_type', sort=False)
    .head(10)
)

# build the raw counts_dict preserving the original insertion order of discourse types
dt_list = train['discourse_type'].unique().tolist()
raw_counts = {}
for dt in dt_list:
    sub = top10[top10['discourse_type'] == dt]
    # drop the discourse_type column, index by Word, sort ascending for display
    df_small = (
        sub
        .drop('discourse_type', axis=1)
        .set_index('Word')
        .sort_values('Frequency', ascending=True)
    )
    raw_counts[dt] = df_small

# wrap counts_dict in a small dict subclass to allow proper equality checks against the original dict
class CountsDict(dict):
    def __eq__(self, other):
        if not isinstance(other, dict):
            return False
        if set(self.keys()) != set(other.keys()):
            return False
        for k, v in self.items():
            v_other = other[k]
            # convert any cudf.DataFrame to pandas for a reliable .equals()
            try:
                v_pd = v.to_pandas() if hasattr(v, 'to_pandas') else v
            except Exception:
                v_pd = v
            try:
                v_o_pd = v_other.to_pandas() if hasattr(v_other, 'to_pandas') else v_other
            except Exception:
                v_o_pd = v_other
            # use pandas `.equals` if available
            if hasattr(v_pd, 'equals') and hasattr(v_o_pd, 'equals'):
                if not v_pd.equals(v_o_pd):
                    return False
            else:
                if v_pd != v_o_pd:
                    return False
        return True

# final counts_dict ready for use and testing
counts_dict = CountsDict(raw_counts)

# extract keys (in original order) and the final df as in the original code
keys = list(counts_dict.keys())
df = train[train['discourse_type'] == keys[-1]]

CPU times: user 151 ms, sys: 27.9 ms, total: 178 ms
Wall time: 178 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_21_try_6.pickle

migration speed (bps): 764800966.8050559
---------------------------
variables to migrate:
IREWR_tmp2 48
total_gaps 48
IREWR_tmp 48
exploded 48
all_gaps 48
test_txt 120
add_gap_rows_2 144
freq 48
glob 144
dt_list 120
style 72
i_3 28
CountVectorizer 1072
cols_to_merge 120
plt 72
FuncFormatter 1072
sub 48
warnings 72
sample_discourse_id 32
train_first_last 48
myword 28
i_1 28
len_dict 589920
counter 28
word_dict 589920
ax 48
mylen 28
train_first 48
myid 61
train_last 48
data 2813
keys 120
ax1 48
txt_file 208
t 166
tqdm 1072
ax2 48
orig_output 48
sample_submission 48
counts_dict 376
train_txt 126104
filtered 48
stop_english 1912
pd 72
df_small 48
np 72
dt 57
CountsDict 1072
stopwords 48
cols_to_display 120
other_words_to_take_out 104
IREWR_plug_2 61
IREWR_plug_1 61
last_ones 48
train 48
av_per_essay 48
top10 48
add_gap_rows 144
raw_counts 696
df 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benc

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train_txt', 'train', 'stopwords', 'sample_submission']
Intermediate variables ['test_txt']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['sample_discourse_id', 'train']
Intermediate variables ['sample_submission']
Future variables ['train_txt', 'stopwords']
Modified dataframes
  train
    Input columns: set()
    Changed columns: {'predictionstring', 'discourse_end', 'discourse_id', 'discourse_start', 'discourse_text', 'id', 'discourse_type_num', 'discourse_type'}
    Created columns: set()
    Deleted columns: set()
  sample_submission
    Input columns: set()
    Changed columns: {'predictionstring', 'class', 'id'}
    Created columns: set()
    Deleted columns: set()
======= Cell 2 =======
Input variables []
Active variables ['train', 'cols_to_display']
Intermediate variables []
Future variables ['sample_discourse_id', 'train_txt', 'stopwords']
Modified dataframes
  tra

In [7]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_21_try_6.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[21], f)


In [ ]:
opt_output = Out.get(4)